# Hands check: can the camera see your hands clearly?

**Run this once, before you record your clips.** It takes about 2 minutes. **Nothing is recorded or saved.**
It opens your camera, draws a dot on every finger joint, and shows what percentage of the time each hand is found.
If the numbers are good, your recordings will be usable. If not, you fix the light or camera position now, not after 30 takes.

## Step by step

1. At the top of Colab click **Runtime**, then **Change runtime type**. Pick the **2025.07** version (Python 3.11). Click **Save**.
   *(If you do not see a version list, skip this. Come back to it only if step 2 or 3 shows a red error.)*
2. Find the cell titled **2. Install** and click the round **play** button on its left. Wait until it says **Done**.
3. When it says to restart, click **Runtime**, then **Restart session**. Click **Yes** if it asks.
4. Find the cell titled **3. Setup** and click its play button. Wait until it says **Ready**.
5. Find the cell titled **4. Run** and click its play button.
6. When the browser asks to use your camera, click **Allow**.
7. Sit exactly where you will record: same chair, same distance from the camera, same light.
8. Hold both hands up. You should see **dots on every finger**. Each hand has a bar and a percentage.
9. Now move your hands the way you will when signing: slowly, quickly, crossed, one over the other, near your face, and near the edge of the picture. **Watch the bars.**
   - **Green** (90% or more) is good.
   - **Yellow** (70 to 90%) is risky.
   - **Red** (below 70%) is bad. Fix it (more light, move back, lower the camera, keep your hands inside the picture), click **Reset totals**, and try again.
10. When both hands stay **green** during your normal signing moves, you are ready. Close this notebook and open your normal collection notebook.

## If something goes wrong

| What you see | What to do |
| --- | --- |
| A red error in step 2 or step 4 | Wrong runtime. Do step 1 again, then run the cells again from step 2. |
| No camera picture | The camera is blocked. Click the camera icon in the browser's address bar, choose **Allow**, then press **Start** on the page. |
| The picture is jumpy or slow | Slow internet. Close other tabs and apps, or sit closer to the router. The percentages still work. |
| You get a message saying NumPy or MediaPipe is the wrong version | Run **2. Install** again, restart the session (step 3), then run **3. Setup** again. |


In [ ]:
#@title 2. Install { display-mode: "form" }
# Installs the exact versions that work together. MediaPipe 0.10.18 needs NumPy 1.x and protobuf 4.x, while a
# fresh Colab ships NumPy 2 and protobuf 5, so these are pinned. No TensorFlow is needed for this notebook.
%pip install -q numpy==1.26.4 protobuf==4.25.9 opencv-python-headless==4.10.0.84 attrs absl-py flatbuffers matplotlib
%pip install -q --no-deps mediapipe==0.10.18
print("\nDone. Now click Runtime > Restart session (Yes if it asks), then run the cell titled '3. Setup'.")


In [ ]:
#@title 3. Setup { display-mode: "form" }
# Loads the hand tracker and checks that the install is correct. Nothing is saved anywhere.
import base64
import json
import sys
import time


def _stop(msg):
    raise RuntimeError("\n\nPROBLEM: " + msg + "\n")


# ---- 1. Check the install (the usual causes of trouble are listed in plain words) ----
_py = sys.version_info
if _py.major != 3 or _py.minor not in (10, 11):
    _stop(f"This is Python {_py.major}.{_py.minor}, but MediaPipe needs 3.10 or 3.11. "
          "Do step 1 again (Runtime > Change runtime type > pick 2025.07 > Save), then run cell 2 again.")

try:
    import numpy as np
    import cv2
    import google.protobuf
    import mediapipe as mp
except ImportError as e:
    _stop(f"Something is not installed ({e}). Run the cell titled '2. Install', then Runtime > "
          "Restart session, then run this cell again.")

if int(np.__version__.split(".")[0]) >= 2:
    _stop(f"NumPy {np.__version__} is too new for MediaPipe (it needs NumPy below 2). Run the cell titled "
          "'2. Install', then Runtime > Restart session, then run this cell again.")
if mp.__version__ != "0.10.18":
    _stop(f"MediaPipe {mp.__version__} is installed, but this notebook needs 0.10.18. Run the cell titled "
          "'2. Install', then Runtime > Restart session, then run this cell again.")
if int(google.protobuf.__version__.split(".")[0]) >= 5:
    _stop(f"protobuf {google.protobuf.__version__} is too new for MediaPipe 0.10.18. Run the cell titled "
          "'2. Install', then Runtime > Restart session, then run this cell again.")
try:
    from google.colab import output as _colab_output
except ImportError:
    _stop("This notebook only works inside Google Colab (colab.research.google.com).")

print(f"python {_py.major}.{_py.minor} | numpy {np.__version__} | protobuf {google.protobuf.__version__} | "
      f"opencv {cv2.__version__} | mediapipe {mp.__version__}")

# ---- 2. The hand tracker (hands only: no pose, no face, so nothing extra is downloaded) ----
FRAME_W, FRAME_H = 640, 360
SAMPLE_FPS = 15


def _make_detector():
    return mp.solutions.hands.Hands(static_image_mode=False, max_num_hands=2, model_complexity=0,
                                    min_detection_confidence=0.5, min_tracking_confidence=0.5)


DETECTOR = _make_detector()


def _reply(d):
    try:
        from IPython.display import JSON
        return JSON(d)
    except ImportError:  # outside a notebook (tests)
        return d


def cb_hc_init():
    return _reply({"ok": True})


def cb_hc_frame(b64):
    """One JPEG frame in, which hands were found out. MediaPipe is given the UN-mirrored picture.
    On an un-mirrored picture MediaPipe's label 'Left' is the person's RIGHT hand, so the labels
    are swapped here to say what the person sees."""
    try:
        buf = np.frombuffer(base64.b64decode(b64), np.uint8)
        bgr = cv2.imdecode(buf, cv2.IMREAD_COLOR)
        if bgr is None:
            return _reply({"error": "could not read the camera picture"})
        res = DETECTOR.process(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    except Exception as e:  # keep the page alive and show the reason
        return _reply({"error": repr(e)})
    hands = []
    if res.multi_hand_landmarks:
        for lm, hd in zip(res.multi_hand_landmarks, res.multi_handedness):
            label = hd.classification[0].label
            hands.append({"side": "right" if label == "Left" else "left",
                          "pts": [[round(p.x, 4), round(p.y, 4)] for p in lm.landmark]})
    return _reply({"right": any(h["side"] == "right" for h in hands),
                   "left": any(h["side"] == "left" for h in hands),
                   "count": len(hands), "hands": hands})


def cb_hc_reset():
    """Stop pressed: start the tracker from scratch so the next Start is clean."""
    global DETECTOR
    try:
        DETECTOR.close()
    except Exception:
        pass
    DETECTOR = _make_detector()
    return _reply({"ok": True})


def register_callbacks():
    _colab_output.register_callback("hc.init", cb_hc_init)
    _colab_output.register_callback("hc.frame", cb_hc_frame)
    _colab_output.register_callback("hc.reset", cb_hc_reset)


def ui_config(mirror, live_overlay, green_at, yellow_at, window_seconds, jpeg_quality):
    conns = sorted([int(a), int(b)] for a, b in mp.solutions.hands.HAND_CONNECTIONS)
    return {"width": FRAME_W, "height": FRAME_H, "sampleFps": SAMPLE_FPS, "jpegQuality": float(jpeg_quality),
            "mirror": bool(mirror), "liveOverlay": bool(live_overlay), "greenAt": float(green_at),
            "yellowAt": float(yellow_at), "windowSeconds": float(window_seconds), "handConn": conns}


# Warm-up so the first real frame is not slow
_blank = np.zeros((FRAME_H, FRAME_W, 3), np.uint8)
_t = time.time()
DETECTOR.process(_blank)
print(f"Ready. Hand tracker loaded ({(time.time() - _t) * 1000:.0f} ms for the first picture).")
print("Next: run the cell titled '4. Run'.")


In [ ]:
#@title 4. Run { display-mode: "form" }
# You do not need to change anything here.
LIVE_PREVIEW_OVERLAY = False  #@param {type:"boolean"}
# False: show the exact picture the tracker analysed, so the dots always sit on your fingers (a little choppy).
# True: show the smooth live video, but the dots trail behind your hands by a fraction of a second.
MIRROR_PREVIEW = True  #@param {type:"boolean"}
GREEN_AT = 90  #@param {type:"number"}
YELLOW_AT = 70  #@param {type:"number"}
WINDOW_SECONDS = 3  #@param {type:"number"}
JPEG_QUALITY = 0.8  #@param {type:"number"}

from IPython.display import display, Javascript

register_callbacks()
APP_JS = r'''
// Hands check page. Camera -> Python (MediaPipe Hands) -> dots + per-hand percentages.
// Nothing is recorded or saved.
(function () {
  const CFG = __CFG__;
  const host = (typeof element !== 'undefined' && element) ? (element[0] || element) : document.body;
  if (window.__hcStop) { try { window.__hcStop(true); } catch (e) {} }

  const W = CFG.width, H = CFG.height, INTERVAL = 1000 / CFG.sampleFps;
  const root = document.createElement('div');
  root.id = 'hc-root';
  root.style.cssText = 'font-family:system-ui,Segoe UI,Arial,sans-serif;max-width:' + (W + 20) + 'px';
  root.innerHTML =
    '<style>' +
    '#hc-root .btn{font-size:15px;padding:8px 16px;margin:6px 8px 6px 0;border-radius:6px;border:1px solid #888;background:#f4f4f4;cursor:pointer}' +
    '#hc-root .row{display:flex;align-items:center;margin:6px 0;font-size:16px}' +
    '#hc-root .name{width:130px;font-weight:600}' +
    '#hc-root .bar{flex:1;height:18px;background:#e0e0e0;border-radius:9px;overflow:hidden;margin:0 10px}' +
    '#hc-root .fill{height:100%;width:0%;background:#bbb;transition:width .15s}' +
    '#hc-root .num{width:190px;font-variant-numeric:tabular-nums}' +
    '</style>' +
    '<div id="hint" style="font-size:22px;font-weight:700;padding:8px 12px;border-radius:8px;background:#eee;color:#444;margin-bottom:8px">Starting camera...</div>' +
    '<canvas id="view" width="' + W + '" height="' + H + '" style="width:100%;max-width:' + W + 'px;background:#000;border-radius:8px;display:block"></canvas>' +
    '<div id="rows" style="margin-top:8px"></div>' +
    '<div id="stat" style="font-size:13px;color:#666;margin:4px 0"></div>' +
    '<div id="warn" style="font-size:14px;color:#b26a00;margin:4px 0"></div>' +
    '<div><button class="btn" id="bStop">Stop</button><button class="btn" id="bStart" style="display:none">Start</button>' +
    '<button class="btn" id="bReset">Reset totals</button></div>' +
    '<div id="msg" style="font-size:14px;color:#444;margin:4px 0"></div>' +
    '<div id="tips" style="font-size:14px;color:#555;margin-top:8px"><b>Try these moves:</b> hands still &middot; slow &middot; fast &middot; ' +
    'crossed &middot; one hand over the other &middot; near your face &middot; at the edge of the picture. ' +
    'Anything that turns a bar red will also hurt your recordings.</div>';
  host.appendChild(root);
  const q = (s) => root.querySelector(s);

  const ROWS = [['right', 'Right hand'], ['left', 'Left hand'], ['both', 'Both hands']];
  q('#rows').innerHTML = ROWS.map(([k, label]) =>
    '<div class="row" id="row-' + k + '"><div class="name">' + label + '</div><div class="bar"><div class="fill"></div></div>' +
    '<div class="num"><span class="now">--</span> now &nbsp;|&nbsp; <span class="tot">--</span> total</div></div>').join('');

  const view = q('#view'), vctx = view.getContext('2d');
  const cap = document.createElement('canvas'); cap.width = W; cap.height = H;
  const cctx = cap.getContext('2d');
  const state = { running: false, stream: null, video: null, hist: [], tot: { n: 0, right: 0, left: 0, both: 0 },
                  last: null, replyT: [], startedAt: 0, raf: 0, token: 0 };
  const sleep = (ms) => new Promise((r) => setTimeout(r, ms));
  const say = (t) => { q('#msg').textContent = t; };

  async function call(name, args) {
    const r = await google.colab.kernel.invokeFunction(name, args || [], {});
    return r.data['application/json'];
  }

  function drawCap() {   // centre-crop the camera picture to W x H, NOT mirrored (this is what MediaPipe sees)
    const v = state.video, vw = v.videoWidth, vh = v.videoHeight;
    const s = Math.max(W / vw, H / vh), sw = W / s, sh = H / s;
    cctx.drawImage(v, (vw - sw) / 2, (vh - sh) / 2, sw, sh, 0, 0, W, H);
  }

  function drawHands(hands) {
    for (const h of hands || []) {
      const col = h.side === 'right' ? '#00e676' : '#ffab00';
      vctx.lineWidth = 3; vctx.strokeStyle = col;
      for (const [a, b] of CFG.handConn) {
        vctx.beginPath(); vctx.moveTo(h.pts[a][0] * W, h.pts[a][1] * H); vctx.lineTo(h.pts[b][0] * W, h.pts[b][1] * H); vctx.stroke();
      }
      for (let i = 0; i < h.pts.length; i++) {
        const tip = (i === 4 || i === 8 || i === 12 || i === 16 || i === 20);
        vctx.beginPath(); vctx.arc(h.pts[i][0] * W, h.pts[i][1] * H, tip ? 6 : 4.5, 0, 6.2832);
        vctx.fillStyle = col; vctx.fill(); vctx.lineWidth = 1.5; vctx.strokeStyle = '#fff'; vctx.stroke();
      }
    }
  }

  function render(source) {   // source: 'cap' (the exact picture MediaPipe analysed) or 'video' (live)
    vctx.save();
    if (CFG.mirror) { vctx.translate(W, 0); vctx.scale(-1, 1); }
    if (source === 'video' && state.video && state.video.videoWidth) {
      const v = state.video, vw = v.videoWidth, vh = v.videoHeight, s = Math.max(W / vw, H / vh), sw = W / s, sh = H / s;
      vctx.drawImage(v, (vw - sw) / 2, (vh - sh) / 2, sw, sh, 0, 0, W, H);
    } else {
      vctx.drawImage(cap, 0, 0);
    }
    drawHands(state.last);
    vctx.restore();
  }

  function liveLoop() {
    if (!state.running) return;
    render('video');
    state.raf = requestAnimationFrame(liveLoop);
  }

  const pct = (arr, k) => arr.length ? (100 * arr.reduce((s, e) => s + (e[k] ? 1 : 0), 0) / arr.length) : null;
  const colour = (p) => p === null ? '#bbb' : (p >= CFG.greenAt ? '#2e7d32' : (p >= CFG.yellowAt ? '#f9a825' : '#c62828'));

  function updateUi() {
    const now = performance.now();
    state.hist = state.hist.filter((e) => now - e.t <= CFG.windowSeconds * 1000);
    const nowP = {}, totP = {};
    for (const [k] of ROWS) {
      nowP[k] = pct(state.hist, k);
      totP[k] = state.tot.n ? 100 * state.tot[k] / state.tot.n : null;
      const row = q('#row-' + k), p = nowP[k];
      row.querySelector('.now').textContent = p === null ? '--' : Math.round(p) + '%';
      row.querySelector('.tot').textContent = totP[k] === null ? '--' : Math.round(totP[k]) + '%';
      const f = row.querySelector('.fill'); f.style.width = (p === null ? 0 : p) + '%'; f.style.background = colour(p);
    }
    // hint line (advice only; the numbers are the real answer)
    const hint = q('#hint');
    let text = 'Hold both hands up in front of the camera.', bg = '#eee', fg = '#444';
    if (state.hist.length >= 8) {
      const r = nowP.right, l = nowP.left, b = nowP.both;
      if (r < 30 && l < 30) { text = 'No hands seen. Raise both hands into the picture and into the light.'; bg = '#ffcdd2'; fg = '#b71c1c'; }
      else if (b >= CFG.greenAt) { text = 'Both hands look good.'; bg = '#c8e6c9'; fg = '#1b5e20'; }
      else if (Math.min(r, l) < CFG.yellowAt) {
        const weak = r < l ? 'right' : 'left';
        text = 'Your ' + weak + ' hand drops out. Keep it inside the picture, in good light, and not hidden behind the other hand.';
        bg = '#ffcdd2'; fg = '#b71c1c';
      } else { text = 'Almost: some frames lose a hand. Try more light, a plainer background, or slower moves.'; bg = '#fff3cd'; fg = '#7a5b00'; }
    }
    hint.textContent = text; hint.style.background = bg; hint.style.color = fg;
  }

  function handle(res) {
    state.last = res.hands || [];
    const e = { t: performance.now(), right: !!res.right, left: !!res.left, both: (res.count || 0) >= 2 };
    state.hist.push(e);
    state.tot.n++; for (const k of ['right', 'left', 'both']) if (e[k]) state.tot[k]++;
    state.replyT.push(e.t); if (state.replyT.length > 20) state.replyT.shift();
    if (!CFG.liveOverlay) render('cap');
    updateUi();
  }

  function updateStat(rt) {
    const t = state.replyT, n = t.length;
    const fps = n > 1 ? (n - 1) / ((t[n - 1] - t[0]) / 1000) : 0;
    q('#stat').textContent = 'frames ' + state.tot.n + '  |  ' + fps.toFixed(1) + ' fps  |  ' + Math.round(rt) + ' ms round trip';
    const slow = performance.now() - state.startedAt > 4000 && n > 3 && fps < 0.8 * CFG.sampleFps;
    q('#warn').textContent = slow ? 'Only ' + fps.toFixed(1) + ' pictures per second reach the tracker (target ' + CFG.sampleFps +
      '). The numbers still work, but the picture will look choppy. Close other tabs or move closer to the router.' : '';
  }

  async function loop(token) {
    let next = performance.now();
    while (state.running && token === state.token) {
      const wait = next - performance.now(); if (wait > 0) await sleep(wait);
      if (!state.running || token !== state.token) break;
      drawCap();
      const b64 = cap.toDataURL('image/jpeg', CFG.jpegQuality).split(',')[1];
      const t0 = performance.now();
      let res;
      try { res = await call('hc.frame', [b64]); }
      catch (e) { await stop(false, true); say('Lost contact with Colab: ' + (e && e.message ? e.message : e) + '. Press Start to try again.'); return; }
      if (!state.running || token !== state.token) break;
      if (res && res.error) { await stop(false, true); say('Error: ' + res.error + '. Press Start to try again.'); return; }
      handle(res || {});
      updateStat(performance.now() - t0);
      next = Math.max(next + INTERVAL, performance.now());
    }
  }

  async function start() {
    if (state.running) return;
    q('#bStart').style.display = 'none'; q('#bStop').style.display = '';
    say('Opening the camera. Click Allow if the browser asks.');
    try {
      state.stream = await navigator.mediaDevices.getUserMedia({ video: { width: { ideal: 1280 }, height: { ideal: 720 }, facingMode: 'user' }, audio: false });
    } catch (e) {
      say('The camera is blocked or busy (' + (e && e.name ? e.name : e) + '). Click the camera icon in the address bar, choose Allow, then press Start.');
      q('#bStart').style.display = ''; q('#bStop').style.display = 'none'; return;
    }
    const v = document.createElement('video'); v.muted = true; v.playsInline = true; v.srcObject = state.stream;
    await v.play().catch(() => {});
    for (let i = 0; i < 100 && !v.videoWidth; i++) await sleep(50);
    state.video = v;
    try { await call('hc.init'); } catch (e) { await stop(false, true); say('Colab is not answering. Did you run the cells titled 3 and 4, in order? (' + (e && e.message ? e.message : e) + ')'); return; }
    state.running = true; state.startedAt = performance.now(); state.replyT = []; state.token++;
    say('Running. Try your signing moves and watch the bars.');
    if (CFG.liveOverlay) liveLoop();
    loop(state.token);
  }

  async function stop(quiet, keepMsg) {
    state.running = false; state.token++;
    if (state.raf) cancelAnimationFrame(state.raf);
    if (state.stream) { state.stream.getTracks().forEach((t) => t.stop()); state.stream = null; }
    q('#bStop').style.display = 'none'; q('#bStart').style.display = '';
    if (!quiet) { try { await call('hc.reset'); } catch (e) {} if (!keepMsg) say('Stopped. Press Start to run again.'); }
  }
  window.__hcStop = (quiet) => stop(quiet);

  function resetTotals() {
    state.hist = []; state.tot = { n: 0, right: 0, left: 0, both: 0 }; state.replyT = []; state.startedAt = performance.now(); updateUi();
    say(state.running ? 'Totals cleared. Try again.' : 'Totals cleared.');
  }

  q('#bStop').onclick = () => stop();
  q('#bStart').onclick = () => start();
  q('#bReset').onclick = () => resetTotals();
  updateUi();
  start();
})();
'''
CFG = ui_config(MIRROR_PREVIEW, LIVE_PREVIEW_OVERLAY, GREEN_AT, YELLOW_AT, WINDOW_SECONDS, JPEG_QUALITY)
print("Starting. Click Allow when the browser asks for your camera. Hold both hands up, then try your signing moves.")
display(Javascript(APP_JS.replace("__CFG__", json.dumps(CFG))))
